In [1]:
# pip install openai

In [54]:
import chromadb

In [4]:
# pip install streamlit

In [3]:
# pip install chromadb

In [6]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [62]:
api_key = "ur_key"

In [56]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "say hello"}]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


In [6]:
from pypdf import PdfReader
reader = PdfReader("../apple_report.pdf")
print(reader.pages[0].extract_text()[:600])

UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended September 30, 2023
or
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from              to             .
Commission File Number: 001-36743
Apple Inc.
(Exact name of Registrant as specified in its charter)
California 94-2404110
(State or other jurisdictionof incorporation or organization) (I.R.S. Employer Identification No.)


In [46]:
def chunk_text(full_text,page_map,chunk_size=500,overlap=50):
    def get_page_num(chunk_st):
        p_no = 1
        for page in page_map:
            if page['starting_word']<=chunk_st:
                p_no = page['page']
            else:
                break
        return p_no 
    
    st_word=0
    raw_words = full_text.split()
    i_d = 0 
    page_dict =[]
    while st_word<=len(raw_words):
        
        end_word = st_word+chunk_size
        chunk_words = " ".join(raw_words[st_word:end_word])
        page_no = get_page_num(st_word)
        st_word= end_word-overlap
        page_dict.append({
            "chunk_id": i_d,
            "text": chunk_words,
            "start_word": st_word,
            "page": page_no
        })
        i_d+=1
    return page_dict

# def chunk_text(full_text, page_map, chunk_size=500, overlap=50):
#     words = full_text.split()
#     chunks = []
    
#     def get_page(start_word):
#         page_num = 1
#         for entry in page_map:
#             if entry["starting_word"] <= start_word:
#                 page_num = entry["page"]
#             else:
#                 break
#         return page_num
    
#     start = 0
#     while start < len(words):
#         end = start + chunk_size
#         chunk_text = " ".join(words[start:end])
#         page = get_page(start)
#         chunks.append({
#             "chunk_id": len(chunks),
#             "text": chunk_text,
#             "start_word": start,
#             "page": page
#         })
#         start = end - overlap
#     return chunks

chunks = chunk_text(full_text, page_map)
print(f"Total chunks: {len(chunks)}")
print(f"Sample: chunk_id={chunks[10]['chunk_id']}, page={chunks[10]['page']}")
        

Total chunks: 91
Sample: chunk_id=10, page=9


In [47]:
def load_pdf_as_single_text(filepath):
    reader = pypdf.PdfReader(filepath)
    full_txt = ""
    page_map = []
    for i,page in enumerate(reader.pages):
        txt = page.extract_text()
        page_map.append({'page':i+1,'starting_word':len(full_txt.split())})
        full_txt+=txt+" "
    return full_txt,page_map
    
full_text, page_map = load_pdf_as_single_text("apple_report.pdf")
print(f"Total characters: {len(full_text)}")
print(f"Page map sample: {page_map[:3]}")

Total characters: 267682
Page map sample: [{'page': 1, 'starting_word': 0}, {'page': 2, 'starting_word': 413}, {'page': 3, 'starting_word': 841}]


In [48]:
import pypdf
# filepath = 'apple_report.pdf'
def load_pdf(filepath):
    reader = pypdf.PdfReader(filepath)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            pages.append({"page": i+1, "text": text})
    return pages
        
pages = load_pdf("apple_report.pdf")
print(f"Total pages loaded: {len(pages)}")
print(pages[0])

Total pages loaded: 80
{'page': 1, 'text': 'UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\nFORM 10-K\n(Mark One)\n☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the fiscal year ended September 30, 2023\nor\n☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the transition period from              to             .\nCommission File Number: 001-36743\nApple Inc.\n(Exact name of Registrant as specified in its charter)\nCalifornia 94-2404110\n(State or other jurisdictionof incorporation or organization) (I.R.S. Employer Identification No.)\nOne Apple Park Way\nCupertino, California 95014\n(Address of principal executive offices) (Zip Code)\n(408) 996-1010\n(Registrant’s telephone number, including area code)\nSecurities registered pursuant to Section 12(b) of the Act:\nTitle of each class Trading symbol(s) Name of each exchange on which registered\nCommon Stock, $0.00001 par

In [49]:
# def load_pdf_as_single_text(filepath):   #--------> no info about page from where the chunk came
#     reader = pypdf.PdfReader(filepath)
#     full_text = ""
#     page_map = []
#     for i, page in enumerate(reader.pages):
#         text = page.extract_text()
#         if text:
#             page_map.append({"page": i+1, "start_char": len(full_text)})
#             full_text += text + " "
#     return full_text, page_map

# full_text, page_map = load_pdf_as_single_text("apple_report.pdf")
# print(f"Total characters: {len(full_text)}")

def load_pdf_as_single_text(filepath):
    reader = pypdf.PdfReader(filepath)
    full_text = ""
    page_map = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            page_map.append({
                "page": i+1, 
                "start_word": len(full_text.split())
            })
            full_text += text + " "
    return full_text, page_map

full_text, page_map = load_pdf_as_single_text("apple_report.pdf")
print(f"Total characters: {len(full_text)}")
print(f"Page map sample: {page_map[:3]}")

Total characters: 267682
Page map sample: [{'page': 1, 'start_word': 0}, {'page': 2, 'start_word': 413}, {'page': 3, 'start_word': 841}]


In [50]:
# def chunk_text(full_text, chunk_size=500, overlap=50):
#     words = full_text.split()
#     chunks = []
#     start = 0
#     while start < len(words):
#         end = start + chunk_size
#         chunk_text = " ".join(words[start:end])
#         chunks.append({
#             "chunk_id": len(chunks),
#             "text": chunk_text,
#             "start_word": start
#         })
#         start = end - overlap
#     return chunks

# chunks = chunk_text(full_text)
# print(f"Total chunks: {len(chunks)}")
# print(f"Sample chunk:\n{chunks[0]['text'][:300]}")


def chunk_text(full_text, page_map, chunk_size=500, overlap=50):
    words = full_text.split()
    chunks = []
    
    def get_page(start_word):
        page_num = 1
        for entry in page_map:
            if entry["start_word"] <= start_word:
                page_num = entry["page"]
            else:
                break
        return page_num
    
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk_text = " ".join(words[start:end])
        page = get_page(start)
        chunks.append({
            "chunk_id": len(chunks),
            "text": chunk_text,
            "start_word": start,
            "page": page
        })
        start = end - overlap
    return chunks

chunks = chunk_text(full_text, page_map)
print(f"Total chunks: {len(chunks)}")
print(f"Sample: chunk_id={chunks[10]['chunk_id']}, page={chunks[10]['page']}")

Total chunks: 91
Sample: chunk_id=10, page=9


In [51]:
def embed_chunks(chunks, client):
    texts = [chunk["text"] for chunk in chunks]
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    for i, chunk in enumerate(chunks):
        chunk["embedding"] = response.data[i].embedding
    return chunks

# chunks1 = embed_chunks(chunks, client)


In [61]:
# print(f"Embedding dimension: {len(chunks[0]['embedding'])}")
# print(f"First 5 values: {chunks[0]['embedding'][:5]}")


In [47]:
import chromadb

# def build_vector_store(chunks):
#     # chroma_client = chromadb.Client()   # in memory 
#     chroma_client = chromadb.PersistentClient(path="./apple_chroma_db")  # in folder 
#     collection = chroma_client.create_collection(name="apple_report")
    
    
#     collection.add(
#         ids=[str(chunk["chunk_id"]) for chunk in chunks],
#         embeddings=[chunk["embedding"] for chunk in chunks],
#         documents=[chunk["text"] for chunk in chunks]
#     )
#     return collection

# collection = build_vector_store(chunks)
# print(f"Stored {collection.count()} chunks in vector store")


def build_vector_store(chunks):
    chroma_client = chromadb.PersistentClient(path="./apple_chroma_db")
    
    try:
        chroma_client.delete_collection("apple_report")
    except:
        pass
    
    collection = chroma_client.create_collection(name="apple_report")
    
    collection.add(
        ids=[str(chunk["chunk_id"]) for chunk in chunks],
        embeddings=[chunk["embedding"] for chunk in chunks],
        documents=[chunk["text"] for chunk in chunks],
        metadatas=[{"page": chunk["page"]} for chunk in chunks]
    )
    return collection

collection1 = build_vector_store(chunks1)
print(f"Stored {collection1.count()} chunks with page metadata")

Stored 91 chunks with page metadata


In [57]:
# def ask(question, collection, client):
#     # embed the question
#     question_embedding = client.embeddings.create(
#         model="text-embedding-3-small",
#         input=[question]
#     ).data[0].embedding
    
#     # find the 3 most relevant chunks
#     results = collection.query(
#         query_embeddings=[question_embedding],
#         n_results=3
#     )
    
#     # build context from retrieved chunks
#     context = "\n\n".join(results["documents"][0])
    
#     # ask GPT with that context
#     response = client.chat.completions.create(
#         model="gpt-3.5-turbo",
#         messages=[
#             {
#                 "role": "system",
#                 "content": "You are a financial analyst. Answer questions using only the context provided. If the answer is not in the context, say so."
#             },
#             {
#                 "role": "user",
#                 "content": f"Context:\n{context}\n\nQuestion: {question}"
#             }
#         ]
#     )
    
#     return response.choices[0].message.content

# # test it
# answer = ask("What are Apple's biggest risk factors?", collection, client)
# print(answer)



def ask(question, collection, client):
    question_embedding = client.embeddings.create(
        model="text-embedding-3-small",
        input=[question]
    ).data[0].embedding
    
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=3
    )
    
    context = "\n\n".join(results["documents"][0])
    pages = [r["page"] for r in results["metadatas"][0]]
    # print(context)
    # print(results['metadatas'])
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {
                "role": "system",
                "content": "You are a financial analyst. Answer questions using only the context provided. If the answer is not in the context, say so."
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}"
            }
        ]
    )
    
    answer = response.choices[0].message.content
    citation = f"📄 Sources: Page {', '.join(str(p) for p in sorted(set(pages)))}"
    
    return answer, citation


client_db = chromadb.PersistentClient(path="./apple_chroma_db")
collection1 = client_db.get_collection("apple_report")

answer, citation = ask("What was Apple's total revenue in 2023?", collection1, client)
print(answer)
print(citation)

Apple's total revenue in 2023 was $383.3 billion.
📄 Sources: Page 21, 23, 37


In [31]:
answer = ask("Summarise the financial statements", collection, client)
print(answer)

The financial statements for Apple Inc. as of September 30, 2023, show net sales of $383.285 million, with a gross margin of $169.148 million. Operating income was $114.301 million, resulting in a net income of $96.995 million. The balance sheet indicates total assets of $109.297 million, including cash and cash equivalents of $29.965 million and marketable securities of $31.59 million. Cash generated from operating activities was $110.543 million, while cash used in investing activities was $3.705 million, and cash used in financing activities was $108.488 million.


In [32]:
answer = ask("What was Apple's total revenue in 2023?", collection, client)
print(answer)

Apple's total revenue in 2023 was $383.3 billion.


In [60]:
answer = ask("Who are Apple's main competitors?", collection1, client)
print(answer)

("Apple's main competitors are companies that have significant technical, marketing, distribution, and other resources, as well as established hardware, software, and service offerings with large customer bases. Some competitors have broader product lines, lower-priced products, and a larger installed base of active devices. Competitors imitate Apple's product features, offer aggressive pricing, and may offer products at little or no profit or even at a loss.", '📄 Sources: Page 5, 6, 9')
